# NBA Points Prop Prediction Model

## Overview
This project builds a machine learning model to predict NBA player point totals and compare them against sportsbook lines. The goal is to identify potential value in over/under betting markets by estimating expected scoring output based on historical performance.

## Objectives
- Predict player points for upcoming games using historical game data
- Compare model predictions to sportsbook lines
- Quantify edge and probability of hitting over/under
- Track model performance against real outcomes

## Data Sources
- NBA Stats API (`nba_api`)
- Historical player game logs (2024–2025 season dataset)
- Derived rolling averages and performance metrics

## Model Approach
- Regression-based model trained on player performance features
- Key features include:
  - Rolling averages (last 5, 10, 20 games)
  - Shooting volume (FGA, FTA)
  - Minutes played
  - Volatility (standard deviation of recent performance)

## Output
- Predicted points
- Edge vs sportsbook line
- Probability of over/under hitting
- Model pick (OVER / UNDER)

## Notes
This project is for educational and analytical purposes only. It is not intended for financial advice or guaranteed betting outcomes.

## Environment Setup and Data Ingestion
This section initializes the environment and prepares the data sources used throughout the project.


#### NBA API Installation
The `nba_api` package is used to retrieve player information and historical game logs directly from official NBA data endpoints.


In [3]:
!pip install -q nba_api

#### Library Imports
Core libraries used in this project:
- `nba_api` for player metadata and game logs
- `pandas` for data manipulation and analysis
- `time` to manage request pacing and avoid API rate limits

In [ ]:
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog
import pandas as pd
import time

#### Sportsbook Data
A CSV file containing sportsbook lines is imported. This dataset includes:
- Player names
- Game dates
- Posted point lines
- Sportsbook source

In [ ]:
from google.colab import files
uploaded = files.upload()

#### Load csv

In [ ]:
import pandas as pd

lines_df = pd.read_csv("sportsbook_lines.csv")
lines_df.columns = lines_df.columns.str.strip()

lines_df["GAME_DATE"] = pd.to_datetime(lines_df["GAME_DATE"])
lines_df["PLAYER_NAME"] = lines_df["PLAYER_NAME"].astype(str).str.strip()

print(lines_df.head())
print(lines_df.columns.tolist())

## Collect Player Game Logs from NBA API

This step builds the raw training dataset by pulling player game logs from the NBA API for the selected season.

### What this cell does
- Retrieves the list of active NBA players
- Loops through players one at a time
- Requests each player’s game log for the 2024–25 season
- Adds successful responses to a master list
- Retries failed requests up to a set limit
- Combines all successful player logs into one dataframe
- Exports the result as `nba_training_data.csv`

### Why a queue and retry system are used
NBA API requests can fail because of:
- request timeouts
- temporary connection issues
- empty player logs
- API instability when many requests are made in sequence

To make the collection process more reliable, this cell uses:
- a queue of players to process
- attempt tracking per player
- retry limits
- short pauses between requests

This helps maximize the number of successful player downloads without stopping the entire process because of a few failures.



In [ ]:
from collections import deque
import time

target_successful_players = 300
max_attempts_per_player = 3
season = "2024-25"

player_queue = deque(players.get_active_players())
attempt_counts = {}
successful_player_ids = set()
all_games = []

successful_players = 0
total_attempts = 0

while player_queue and successful_players < target_successful_players:
    player = player_queue.popleft()
    player_id = player["id"]
    player_name = player["full_name"]

    if player_id in successful_player_ids:
        continue

    attempt_counts[player_id] = attempt_counts.get(player_id, 0) + 1
    total_attempts += 1

    print(
        f"Attempt {total_attempts} | Success {successful_players}/{target_successful_players} | "
        f"Try {attempt_counts[player_id]}/{max_attempts_per_player} - {player_name}"
    )

    try:
        gamelog = playergamelog.PlayerGameLog(
            player_id=player_id,
            season=season,
            timeout=10
        )

        df_player = gamelog.get_data_frames()[0]

        if not df_player.empty:
            df_player["PLAYER_NAME"] = player_name
            all_games.append(df_player)
            successful_player_ids.add(player_id)
            successful_players += 1
            print(f"Added {player_name} ({successful_players}/{target_successful_players})")
        else:
            print(f"Empty game log for {player_name}")
            if attempt_counts[player_id] < max_attempts_per_player:
                player_queue.append(player)

        time.sleep(0.1)

    except Exception as e:
        print(f"Failed {player_name}: {e}")
        if attempt_counts[player_id] < max_attempts_per_player:
            player_queue.append(player)
        else:
            print(f"Gave up on {player_name} after {max_attempts_per_player} attempts")

print(f"\nDone.")
print(f"Successful players: {successful_players}")
print(f"Total attempts: {total_attempts}")
print(f"Remaining in queue: {len(player_queue)}")

all_games = [g for g in all_games if not g.empty]
df = pd.concat(all_games, ignore_index=True)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()


df.to_csv("nba_training_data.csv", index=False)

### Interpretation of the output
The output log shows the progress of the collection process in real time.

Examples of what the messages mean:
- `Success X/300` shows how many player logs have been successfully added so far
- `Try 1/3`, `Try 2/3`, etc. shows the current retry attempt for that player
- `Added [player name]` means the player’s game log was successfully retrieved and stored
- `Empty game log` means no usable game log was returned for that player
- `Read timed out` indicates the API did not respond in time for that request
- `Gave up on [player name] after 3 attempts` means all retries failed and the player was skipped

Overall, this output indicates that the dataset collection process is functioning as expected. Some failures are normal when working with live API requests, especially across many players. The retry system helps recover from temporary issues while keeping the data pipeline moving.

### Result
At the end of this step, the project has a combined raw dataset of player game logs that can be cleaned, engineered, and merged with sportsbook data in later sections.

## Combine Player Data with Sportsbook Lines

This step merges the collected player game logs with sportsbook betting lines to create a unified dataset for modeling.

### What this cell does
- Combines all individual player game logs into a single dataframe
- Converts `GAME_DATE` to a proper datetime format for consistency
- Cleans `PLAYER_NAME` to ensure reliable matching
- Performs a **left join** with the sportsbook dataset using:
  - `PLAYER_NAME`
  - `GAME_DATE`

### Why a left join is used
A left join ensures that:
- All player game logs are preserved
- Sportsbook lines are added only where a match exists
- No player performance data is lost due to missing betting lines



In [ ]:
# Combine all player game logs into one dataframe
df = pd.concat(all_games, ignore_index=True)

df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df["PLAYER_NAME"] = df["PLAYER_NAME"].astype(str).str.strip()

df = df.merge(
    lines_df,
    on=["PLAYER_NAME", "GAME_DATE"],
    how="left"
)

print(df[["PLAYER_NAME", "GAME_DATE", "sportsbook_line"]].head(10))

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

### Interpretation of the output

The preview shows:
- Player game logs successfully combined into a single dataset
- Core performance stats (PTS, FGA, MIN, etc.) alongside `sportsbook_line`

You will notice many `NaN` values in the `sportsbook_line` column. This is expected because:
- The sportsbook dataset only contains specific games (e.g., recent or selected games)
- Most historical game logs do not have corresponding betting lines

Key takeaways:
- The merge is working correctly
- Matching occurs only when both player name and game date align
- Missing sportsbook values do not indicate an error — they reflect incomplete coverage of betting data

### Dataset size
- Rows: represents total player-game observations collected
- Columns: includes both raw performance metrics and merged sportsbook data

### Result
The dataset is now structured for feature engineering and modeling, with both player performance data and betting lines aligned where available.

## Prepare and Sort Dataset

This step organizes the dataset to ensure it is properly structured for time-based analysis and feature engineering.

### What this cell does
- Converts `GAME_DATE` to a datetime format (if not already)
- Sorts the dataset by:
  - `PLAYER_NAME`
  - `GAME_DATE` (chronological order)
- Resets the index after sorting

### Why this matters
Time order is critical for this project because:
- Rolling averages (last 5, 10, 20 games) depend on past performance
- Future games must not influence past data (avoids data leakage)
- Each player’s performance needs to be evaluated in sequence

Sorting ensures that:
- All games for each player are grouped together
- Each player’s games are in correct chronological order



In [ ]:
# Prepare and sort the dataset
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df = df.sort_values(["PLAYER_NAME", "GAME_DATE"]).reset_index(drop=True)

df.head()

### Interpretation of the output
The preview shows:
- Data grouped by player (e.g., A.J. Lawson)
- Games ordered from earliest to latest
- Clean indexing after sorting

The presence of `NaN` in `sportsbook_line` is still expected at this stage, since not all games have associated betting lines.

### Result
The dataset is now properly ordered and ready for feature engineering, where rolling statistics and derived metrics will be calculated.

## Create Game Score Feature

This step introduces a composite performance metric called **Game Score**, which summarizes a player’s overall contribution in a single game.

### What this cell does
- Calculates `gmsc` (Game Score) using a weighted formula based on box score statistics
- Combines scoring, shooting efficiency, rebounding, playmaking, and defensive contributions
- Penalizes missed shots, fouls, and turnovers

### Formula components
The Game Score calculation includes:
- Positive contributions:
  - Points (PTS)
  - Field goals made (FGM)
  - Rebounds (OREB, DREB)
  - Assists (AST)
  - Steals (STL)
  - Blocks (BLK)
- Negative contributions:
  - Missed shots (FGA, FTA - FTM)
  - Personal fouls (PF)
  - Turnovers (TOV)

This creates a balanced measure of overall game impact, not just scoring.

### Why this feature is useful
- Captures overall performance beyond points alone
- Helps identify high-impact games that may not be reflected in scoring totals
- Can improve model understanding of player form and efficiency



In [ ]:
# Calculate Game Score
df["gmsc"] = (
    df["PTS"]
    + 0.4 * df["FGM"]
    - 0.7 * df["FGA"]
    - 0.4 * (df["FTA"] - df["FTM"])
    + 0.7 * df["OREB"]
    + 0.3 * df["DREB"]
    + df["STL"]
    + 0.7 * df["AST"]
    + 0.7 * df["BLK"]
    - 0.4 * df["PF"]
    - df["TOV"]
)

df[["PLAYER_NAME", "GAME_DATE", "PTS", "gmsc"]].head()

### Interpretation of the output
The preview shows:
- `PTS` (raw scoring output)
- `gmsc` (overall performance score)

Examples:
- A low or negative `gmsc` indicates poor efficiency or limited contribution
- A higher `gmsc` reflects strong all-around performance

This helps contextualize scoring by showing how efficiently and effectively those points were achieved.

### Result
The dataset now includes a more comprehensive performance metric that can be used alongside points in feature engineering and modeling.

## Generate Rolling Features (Model Inputs)

This step creates the core features used by the model. These features summarize a player’s recent performance and context leading into each game.

### Key Principle: No Data Leakage
All features are calculated using **only prior games** by applying `.shift(1)` before rolling calculations.

This ensures:
- The model never uses future information
- Each prediction reflects what was known *before* the game occurred

---

### Feature Groups

#### 1. Scoring Trends
- `last5_pts`, `last10_pts`, `last3_pts`
- Rolling averages of recent scoring performance

These capture short-term and medium-term scoring form.

---

#### 2. Overall Performance (Game Score)
- `last5_gmsc`, `last10_gmsc`

Provides a broader measure of player impact beyond just points.

---

#### 3. Playing Time & Efficiency
- `last5_minutes`
- `last5_fg_pct`

Tracks:
- Opportunity (minutes played)
- Efficiency (shooting performance)

---

#### 4. Game Context
- `home_game`: whether the game is played at home
- `days_rest`: days since last game
- `is_back_to_back`: indicator for fatigue (1-day rest)

These help capture situational effects on performance.

---

#### 5. Usage & Role
- `usage_proxy = FGA + 0.44 * FTA + TOV`
- `last5_usage_proxy`

Estimates how involved a player is in the offense.

Higher usage → more scoring opportunity.

---

#### 6. Season Baseline
- `season_minutes_avg`

Expanding average of minutes played across the season, giving a stable long-term baseline.

---

#### 7. Volatility (Consistency)
- `minutes_volatility`
- `points_volatility`

Standard deviation of recent performance:
- High volatility = inconsistent player
- Low volatility = stable output

---

#### 8. Optional Shooting Feature
- `last5_3pa` (if available)

Captures recent three-point attempt volume.

---


In [ ]:
# Create rolling features using only prior games
df["last5_pts"] = df.groupby("PLAYER_NAME")["PTS"].transform(lambda x: x.shift(1).rolling(5).mean())
df["last10_pts"] = df.groupby("PLAYER_NAME")["PTS"].transform(lambda x: x.shift(1).rolling(10).mean())

df["last5_gmsc"] = df.groupby("PLAYER_NAME")["gmsc"].transform(lambda x: x.shift(1).rolling(5).mean())
df["last10_gmsc"] = df.groupby("PLAYER_NAME")["gmsc"].transform(lambda x: x.shift(1).rolling(10).mean())

df["last5_minutes"] = df.groupby("PLAYER_NAME")["MIN"].transform(lambda x: x.shift(1).rolling(5).mean())
df["last5_fg_pct"] = df.groupby("PLAYER_NAME")["FG_PCT"].transform(lambda x: x.shift(1).rolling(5).mean())

df["home_game"] = df["MATCHUP"].str.contains("vs").astype(int)

df["days_rest"] = df.groupby("PLAYER_NAME")["GAME_DATE"].diff().dt.days
df["days_rest"] = df["days_rest"].fillna(3)

df["is_back_to_back"] = (df["days_rest"] == 1).astype(int)

df["usage_proxy"] = df["FGA"] + 0.44 * df["FTA"] + df["TOV"]

df["season_minutes_avg"] = df.groupby("PLAYER_NAME")["MIN"].transform(lambda x: x.shift(1).expanding().mean())

df["last3_pts"] = df.groupby("PLAYER_NAME")["PTS"].transform(lambda x: x.shift(1).rolling(3).mean())

df["last5_usage_proxy"] = df.groupby("PLAYER_NAME")["usage_proxy"].transform(lambda x: x.shift(1).rolling(5).mean())
df["minutes_volatility"] = df.groupby("PLAYER_NAME")["MIN"].transform(lambda x: x.shift(1).rolling(5).std())
df["points_volatility"] = df.groupby("PLAYER_NAME")["PTS"].transform(lambda x: x.shift(1).rolling(5).std())

if "FG3A" in df.columns:
    df["last5_3pa"] = df.groupby("PLAYER_NAME")["FG3A"].transform(lambda x: x.shift(1).rolling(5).mean())

df[
    [
        "PLAYER_NAME",
        "GAME_DATE",
        "PTS",
        "last5_pts",
        "last10_pts",
        "last5_gmsc",
        "last10_gmsc",
        "last5_minutes",
        "last5_fg_pct"
    ]
].head(12)

### Interpretation of the Output

The preview shows how features evolve over time for a player:

- Early rows contain `NaN` values → expected because there are not enough prior games to compute rolling averages
- As more games accumulate:
  - Rolling features begin to populate
  - Values stabilize and become more informative

Example progression:
- First few games → no history → `NaN`
- After ~5 games → `last5_*` features appear
- After ~10 games → `last10_*` features appear

This confirms that:
- Features are being calculated correctly
- Only past data is being used
- The dataset is becoming model-ready

---

### Result

The dataset now includes time-aware, player-specific features that capture:
- Recent performance trends
- Usage and opportunity
- Game context and fatigue
- Consistency and volatility

These features form the foundation of the prediction model.

---

## Final Feature Engineering and Model Dataset Preparation

This step expands the feature set and prepares the final dataset used for training the prediction model.

### What this cell does

#### 1. Additional Feature Engineering

This section adds deeper signals to improve model performance:

- Turnovers
  - `last5_tov`: recent turnover trends

- Shot Volume
  - `last5_fga`, `last10_fga`: recent shot attempts

- Free Throw Activity
  - `last5_fta`: recent free throw attempts

- Usage Proxy
  - `last5_usage_proxy = FGA + FTA + TOV`
  - Represents offensive involvement

#### 2. Player Baseline Scoring

- `player_avg_pts`: expanding average of points
- `player_avg_pts_sq`: squared version to capture nonlinear effects

This helps the model understand:
- A player’s typical scoring level
- Differences between role players and high-volume scorers

#### 3. Extended Time-Based Features

- `last10_minutes`: medium-term playing time trend
- `last3_pts`, `last5_pts`, `last10_pts`, `last20_pts`: multi-window scoring trends

These provide:
- Short-term form
- Medium-term consistency
- Long-term baseline

#### 4. Volatility Features

- `minutes_volatility`
- `points_volatility`

These measure consistency:
- High volatility means unpredictable performance
- Low volatility means more stable output

#### 5. Context Features

- `home_game`: home vs away indicator
- `days_rest`: time between games
- `is_back_to_back`: fatigue indicator
- `season_minutes_avg`: long-term playing time baseline

#### 6. Optional Shooting Feature

- `last5_3pa` if available

This captures recent three-point attempt volume.

### Dropping Incomplete Rows

Rows are removed if they are missing key features required by the model.

### Why this is necessary

Early-season or early-career games:
- do not have enough history for rolling features
- would introduce noise or incomplete signals

By dropping these rows:
- the model trains only on fully populated feature sets
- data quality and consistency are improved



In [ ]:
# Turnover proxy
df["last5_tov"] = df.groupby("PLAYER_NAME")["TOV"].transform(
    lambda x: x.shift(1).rolling(5).mean()
)

# Shot volume
df["last5_fga"] = df.groupby("PLAYER_NAME")["FGA"].transform(
    lambda x: x.shift(1).rolling(5).mean()
)

df["last10_fga"] = df.groupby("PLAYER_NAME")["FGA"].transform(
    lambda x: x.shift(1).rolling(10).mean()
)

# Free throw attempts
df["last5_fta"] = df.groupby("PLAYER_NAME")["FTA"].transform(
    lambda x: x.shift(1).rolling(5).mean()
)

# Usage proxy
df["last5_usage_proxy"] = df["last5_fga"] + df["last5_fta"] + df["last5_tov"]

# Player scoring baseline
df["player_avg_pts"] = df.groupby("PLAYER_NAME")["PTS"].transform(
    lambda x: x.shift(1).expanding().mean()
)

# STRONGER baseline signal
df["player_avg_pts_sq"] = df["player_avg_pts"] ** 2

# Minutes trend
df["last10_minutes"] = df.groupby("PLAYER_NAME")["MIN"].transform(
    lambda x: x.shift(1).rolling(10).mean()
)

# Add missing features for model consistency
df["home_game"] = df["MATCHUP"].str.contains("vs").astype(int)

df["days_rest"] = df["GAME_DATE"].diff().dt.days
df["days_rest"] = df["days_rest"].fillna(3)

df["is_back_to_back"] = (df["days_rest"] == 1).astype(int)

df["season_minutes_avg"] = df.groupby("PLAYER_NAME")["MIN"].transform(
    lambda x: x.shift(1).expanding().mean()
)

df["last3_pts"] = df.groupby("PLAYER_NAME")["PTS"].transform(
    lambda x: x.shift(1).rolling(3).mean()
)

df["last5_pts"] = df.groupby("PLAYER_NAME")["PTS"].transform(
    lambda x: x.shift(1).rolling(5).mean()
)

df["last10_pts"] = df.groupby("PLAYER_NAME")["PTS"].transform(
    lambda x: x.shift(1).rolling(10).mean()
)

# Help with star player baseline
df["last20_pts"] = df.groupby("PLAYER_NAME")["PTS"].transform(
    lambda x: x.shift(1).rolling(20).mean()
)

df["minutes_volatility"] = df.groupby("PLAYER_NAME")["MIN"].transform(
    lambda x: x.shift(1).rolling(5).std()
)

df["points_volatility"] = df.groupby("PLAYER_NAME")["PTS"].transform(
    lambda x: x.shift(1).rolling(5).std()
)

# Sportsbook line - Turn on when more data is collected.
#df = df.dropna(subset=["sportsbook_line"])

# Optional 3PA
if "FG3A" in df.columns:
    df["last5_3pa"] = df.groupby("PLAYER_NAME")["FG3A"].transform(
        lambda x: x.shift(1).rolling(5).mean()
    )

# Drop rows missing key features
df_features = df.dropna(subset=[
    "player_avg_pts",
    "player_avg_pts_sq",
    "last5_pts",
    "last10_pts",
    "last20_pts",
    "last5_fga",
    "last5_fta",
    "last5_minutes",
    "last5_gmsc"
]).reset_index(drop=True)

# Preview
df_features.head()

### Interpretation of the output

The preview shows:
- a fully engineered feature set
- no missing values in critical model columns
- rich player-level signals across scoring, usage, efficiency, and context

Key observations:
- features like `player_avg_pts` and `last20_pts` provide strong baseline signals
- rolling features reflect recent trends leading into each game
- volatility metrics add information about consistency

### Result

The dataset `df_features` is now:
- clean
- time-aware
- feature-rich
- free of missing critical values

It is ready for model training and evaluation.

## Define Features and Target

This step defines the input features (`X`) and prediction target (`y`) for the model.

### What this cell does

#### 1. Select Feature Columns

The model uses a mix of:

- Baseline scoring
  - player_avg_pts
  - player_avg_pts_sq

- Opportunity / playing time
  - season_minutes_avg
  - last5_minutes
  - last5_usage_proxy

- Game context
  - home_game
  - days_rest
  - is_back_to_back

- Recent performance trends
  - last3_pts
  - last5_pts
  - last10_pts
  - last20_pts

- Shot volume
  - last5_fga
  - last5_fta

- Performance quality
  - last5_gmsc

- Consistency
  - minutes_volatility
  - points_volatility

- Optional
  - last5_3pa (if available)

---

#### 2. Create Model Dataset

- model_df → clean dataset copy
- X → selected feature columns
- y_points → target (PTS)

This creates a supervised learning setup:

- Inputs → player/game features  
- Output → points scored  

---



In [ ]:
feature_columns = [
    "player_avg_pts",
    "player_avg_pts_sq",
    "season_minutes_avg",
    "home_game",
    "days_rest",
    "is_back_to_back",
    "last3_pts",
    "last5_pts",
    "last10_pts",
    "last20_pts",
    "last5_fga",
    "last5_fta",
    "last5_minutes",
    "last5_gmsc",
    "last5_usage_proxy",
    "minutes_volatility",
    "points_volatility"
]

if "FG3A" in df.columns:
    feature_columns.append("last5_3pa")

model_df = df_features.copy()

X = model_df[feature_columns]
y_points = model_df["PTS"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y_points.shape)

X.head()

## Output Interpretation

Feature matrix shape: (10111, 18)  
Target shape: (10111,)  

This tells us:

- ~10,000 training samples (games)
- 18 predictive features
- Each row = one player-game instance

---

### Why this matters

This is one of the most important steps in the pipeline.

- The model can only learn from these features
- Feature quality directly determines model performance
- Good features → strong predictions  
- Weak features → noisy predictions  

---

### Key insight

This feature set intentionally balances:

- Long-term skill → player_avg_pts  
- Short-term form → rolling stats  
- Opportunity → minutes + usage  
- Context → rest + home/away  
- Stability → volatility metrics  

This allows the model to answer:

> "Given who this player is, how they've been playing, and the context of this game — how many points should we expect?"

---

### Result

You now have:

- X → feature matrix  
- y_points → prediction target  

Ready for model training.

## Split Training and Testing Data

This step divides the dataset into training and testing sets so the model can be evaluated on unseen data.

### What this cell does

- Uses `train_test_split` from sklearn
- Splits data into:
  - X_train, y_train → used to train the model
  - X_test, y_test → used to evaluate performance

Parameters used:
- test_size = 0.2 → 20% of data reserved for testing
- random_state = 42 → ensures reproducibility

---



In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_points,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

### Output Interpretation

X_train shape: (8088, 18)  
X_test shape: (2023, 18)  
y_train shape: (8088,)  
y_test shape: (2023,)  

This tells us:

- ~8,000 training samples
- ~2,000 testing samples
- Each sample has 18 features
- Target values align correctly with feature rows

---

### Why this matters

Without a proper train/test split:

- The model could memorize the data instead of learning patterns
- Performance metrics would be misleading

By holding out test data:

- We simulate real-world predictions
- We measure how well the model generalizes

---

### Important Note - Future Planning

This split is random, not time-based.

That means:
- The model may train on future games and test on past games
- This is fine for initial modeling
- But not ideal for production betting scenarios

A future improvement would be:
- Time-based split (train on past, test on future)

---


## Train and Evaluate Points Regression Model

This step trains the machine learning model and evaluates how well it predicts player points.

### What this cell does

#### 1. Model Selection

A Gradient Boosting Regressor is used:

- n_estimators = 200 → number of trees
- learning_rate = 0.05 → step size for learning
- max_depth = 3 → controls model complexity
- random_state = 42 → reproducibility

Why this model:
- Handles nonlinear relationships well
- Performs strongly on structured/tabular data
- Captures interactions between features

---

#### 2. Model Training

The model is trained using:

- X_train → feature inputs
- y_train → actual points scored

The model learns patterns like:
- how usage impacts scoring
- how recent performance affects outcomes
- how context (rest, home/away) influences results

---

#### 3. Generate Predictions

- y_pred = model predictions on X_test

These represent:
- expected points for each player-game in the test set

---

#### 4. Model Evaluation

MAE: 4.95  
RMSE: 6.30  
R²: 0.50  

---



In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Train the regression model
rf_points = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

rf_points.fit(X_train, y_train)

# Generate predictions
y_pred = rf_points.predict(X_test)

# Evaluate model performance
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R^2:", r2_score(y_test, y_pred))

### Interpretation of Results

#### MAE (Mean Absolute Error) ≈ 4.95

- On average, predictions are off by ~5 points
- This is a solid baseline for NBA props

#### RMSE (Root Mean Squared Error) ≈ 6.30

- Penalizes larger errors more heavily
- Indicates occasional bigger misses (which is expected in sports data)

#### R² Score ≈ 0.50

- Model explains ~50% of variance in scoring
- This is strong for a noisy domain like sports

---

### What this means in practice

- The model is capturing real signal (not random)
- Predictions are useful but not perfect
- There is still room for improvement (matchups, defense, injuries, etc.)

---

### Key Insight

Scoring in the NBA is inherently volatile.

Even a well-performing model:
- cannot perfectly predict outlier performances
- must balance signal vs randomness

An R² around 0.5 in this context is actually meaningful.

---




## Save Model and Test Prediction

This step saves the trained model and performs a simple sanity check on a single prediction.

### What this cell does

#### 1. Save the Trained Model

- Creates a `models` directory if it does not already exist
- Saves the trained model as:

  models/points_regression.pkl

This allows the model to be:
- reused without retraining
- loaded into applications (e.g., Streamlit app)
- shared and version-controlled

---

#### 2. Single Prediction Check

A single sample from the test set is selected:

- sample → one player-game instance
- predicted_points → model prediction
- actual_points → real observed value

Example output:

Predicted points: 20.24  
Actual points: 16  

---



In [ ]:
import joblib
import os

# Create models folder if it does not exist
os.makedirs("models", exist_ok=True)

# Save the trained points regression model
joblib.dump(rf_points, "models/points_regression.pkl")

# Test a single prediction
sample = X_test.iloc[[0]]

predicted_points = rf_points.predict(sample)[0]
actual_points = y_test.iloc[0]

print("Predicted points:", round(predicted_points, 2))
print("Actual points:", actual_points)

### Interpretation

This comparison provides a quick validation that:

- the model is functioning correctly
- predictions are being generated as expected
- output values are in a realistic range

The difference between predicted and actual points (~4 points in this case) is consistent with earlier evaluation metrics.

---

### Why this matters

- Confirms the model pipeline is complete and working end-to-end
- Verifies that saved models can produce valid predictions
- Demonstrates how predictions compare to real outcomes

---

### Result

The model is now:

- trained  
- saved  
- validated on real data  

and ready for use in downstream applications.

## Predicted vs Actual on Test Set

This step evaluates how the model performs by directly comparing predicted points to actual outcomes.

### What this cell does

#### 1. Generate Predictions

- Uses the trained model to predict points on the test set:
  - `test_predictions = model.predict(X_test)`

#### 2. Build Results Dataset

A results dataframe is created containing:

- actual_points → true values from the dataset  
- predicted_points → model predictions  
- prediction_error → predicted - actual  
- abs_error → absolute value of error  

This allows for detailed inspection of model performance at the individual prediction level.

---

#### 3. Summary Metrics

Average absolute error: 4.95  
Median absolute error: 4.01  

These reinforce earlier evaluation:

- Average error ≈ 5 points  
- Median error slightly lower → most predictions are fairly close  

---

#### 4. Best Predictions

The model’s most accurate predictions are displayed:

- Very small errors (near zero)
- Predicted values closely match actual outcomes

This shows:
- The model can be highly precise in favorable situations
- Certain player/game contexts are very predictable

---

#### 5. Worst Predictions

The largest errors are also shown:

- Bigger gaps between predicted and actual points
- Typically caused by:
  - outlier performances
  - unexpected minutes changes
  - game script variability

---



In [ ]:
# Predicted vs Actual on test set
test_predictions = rf_points.predict(X_test)

results_df = X_test.copy()
results_df["actual_points"] = y_test.values
results_df["predicted_points"] = test_predictions
results_df["prediction_error"] = results_df["predicted_points"] - results_df["actual_points"]
results_df["abs_error"] = results_df["prediction_error"].abs()

print("Average absolute error:", round(results_df["abs_error"].mean(), 2))
print("Median absolute error:", round(results_df["abs_error"].median(), 2))

print("\nBest predictions:")
display(
    results_df[
        ["actual_points", "predicted_points", "prediction_error", "abs_error"]
    ].sort_values("abs_error").head(10)
)

print("\nWorst predictions:")
display(
    results_df[
        ["actual_points", "predicted_points", "prediction_error", "abs_error"]
    ].sort_values("abs_error", ascending=False).head(10)
)

### Interpretation

This breakdown highlights two key realities:

#### 1. The model captures real signal

- Many predictions are very close to actual results  
- Indicates strong underlying feature relationships  

#### 2. NBA performance is inherently volatile

- Even good models will miss on outliers  
- Large errors are unavoidable in some cases  

---

### Key Insight

The difference between average and median error is important:

- Median (~4 points) → typical prediction quality  
- Mean (~5 points) → influenced by occasional large misses  

This suggests:
- Most predictions are reasonably accurate  
- A smaller number of games drive larger errors  

---

### Result

This step provides a clear view of:

- how accurate the model is  
- where it performs well  
- where it struggles  

It validates that the model is useful, while also highlighting the natural uncertainty in player scoring.

## Predicted vs Actual Scatter Plot

This visualization compares predicted points to actual points for all test samples.

### What this chart shows

- Each dot represents one player-game instance  
- X-axis → actual points scored  
- Y-axis → predicted points from the model  

The dashed diagonal line represents perfect predictions:
- Points on the line → exact predictions  
- Points above the line → model overestimated  
- Points below the line → model underestimated  

---



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
plt.scatter(results_df["actual_points"], results_df["predicted_points"], alpha=0.6)
plt.xlabel("Actual Points")
plt.ylabel("Predicted Points")
plt.title("Predicted vs Actual Points")

min_val = min(results_df["actual_points"].min(), results_df["predicted_points"].min())
max_val = max(results_df["actual_points"].max(), results_df["predicted_points"].max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.show()

### Interpretation

#### 1. Overall Pattern

There is a clear upward trend:
- As actual points increase, predicted points also increase  
- This indicates the model is capturing real scoring relationships  

---

#### 2. Spread Around the Line

- Points are clustered around the diagonal, but with noticeable spread  
- This reflects prediction error (~5 points on average)  

Closer to the line:
- more accurate predictions  

Further from the line:
- larger errors  

---

#### 3. Mid-Range Accuracy

The model performs best in the mid-range (roughly 10–25 points):

- Highest density of data points  
- Predictions tend to stay closer to the diagonal  
- Most stable and predictable scoring range  

---

#### 4. High-Scoring Games

At higher point totals (30+):

- Increased spread  
- More underprediction  
- Fewer data points  

This suggests:
- High-scoring performances are harder to predict  
- Likely influenced by factors not captured in the model  

---

#### 5. Low-Scoring Games

At very low point totals:

- Slight overprediction tendency  
- Model regresses toward a baseline expectation  

---

### Key Insight

The model behaves like a typical regression model:

- Strong central tendency (pull toward average performance)  
- Difficulty capturing extreme outcomes  
- Best performance in common scoring ranges  

---

### What this confirms

- The model is not random  
- Predictions follow real structure in the data  
- Errors are consistent with earlier metrics (MAE ≈ 5)  

---

### Result

This chart provides a visual validation that:

- the model is learning meaningful patterns  
- predictions are directionally correct  
- variability in outcomes is expected and visible

In [ ]:
from google.colab import files
files.download("models/points_regression.pkl")

In [ ]:
#Estimate prediction uncertainty from test residuals
residuals = y_test - y_pred
points_std = residuals.std()

print("Points prediction std dev:", points_std)

In [ ]:
import json
import os

os.makedirs("models", exist_ok=True)

with open("models/points_model_stats.json", "w") as f:
   json.dump({"std_dev": float(points_std)}, f)